## Hardware Setup:

1) Install CircuitPython on Feather board: https://circuitpython.org/board/adafruit_feather_rp2350/
2) Install drivers for LSM6DS3TR-C and LIS3MDL: https://learn.adafruit.com/adafruit-lsm6ds3tr-c-lis3mdl-precision-9-dof-imu/python-circuitpython
3) Copy code from read_imu.py into code.py file: https://github.com/zacmanchester/spacecraft-attitude-course/blob/main/mekf/read_imu.py


In [ ]:
import Pkg; Pkg.activate(@__DIR__); Pkg.instantiate()

In [ ]:
using LinearAlgebra

In [ ]:
using MeshCat, GeometryBasics, CoordinateTransformations, Rotations
vis = Visualizer()
setobject!(vis[:box1], Rect(Vec(-1.5, -1, -0.5), Vec(3, 2, 1)))

In [ ]:
#quaternion functions
function hat(v)
    return [0 -v[3] v[2];
            v[3] 0 -v[1];
            -v[2] v[1] 0]
end

function unhat(S)
    return 0.5*[S[3,2]-S[2,3];
                S[1,3]-S[3,1];
                S[2,1]-S[1,2]]
end

H = [zeros(1,3); I];
T = [1  zeros(1,3);
     zeros(3,1) -I];

function L(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I + hat(q[2:4])]
end

function R(q)
    return [q[1]          -q[2:4]';
            q[2:4]    q[1]*I - hat(q[2:4])]
end

function G(q)
    return L(q)*H
end

function Q(q)
    return H'*(R(q)'*L(q))*H
end

function expq(ϕ)
    θ = norm(ϕ)
    return [cos(θ); ϕ*sinc(θ/π)];
end

function logq(q)
    θ = acos(q[1])
    r = q[2:4]/norm(q[2:4])
    return θ*r
end

In [ ]:
h = 0.05 #time step

In [ ]:
function state_prediction(x,u,h)
    q = x[1:4]
    β = x[5:7]
    ω = u-β
    Δq = expq(0.5*h*ω)
    return [L(q)*Δq; β]
end

function state_prediction_deriv(x,u,h)
    q = x[1:4]
    β = x[5:7]
    ω = u-β
    Δq = expq(0.5*h*ω)
    qn = L(q)*Δq
    A = [G(qn)'*R(Δq)*G(q) -0.5*h*G(qn)'*G(q);
         zeros(3,3)            I(3)]
    return A
end

In [ ]:
function measurement_prediction(x,r_N)
    q = x[1:4]
    Qk = Q(q)
    y = Qk'*r_N
    return y[:]
end

function measurement_prediction_deriv(x,r_N)
    q = x[1:4]
    m = size(r_N,2)
    C = zeros(3*m,6)
    for k = 1:m
        C[((k-1)*3).+(1:3),1:3] .= H'*(L(q)'*L(H*r_N[:,k]) + R(q)*R(H*r_N[:,k])*T)*G(q)
    end
    return C
end

In [ ]:
using LibSerialPort

In [ ]:
serial_port = "/dev/tty.usbmodem101" #fill this in with correct port
baud_rate = 9600;

In [ ]:
function parse_imu_line(line::String)
    pattern = r"[GAM]:\s*([-\d.]+),\s*([-\d.]+),\s*([-\d.]+)"
    matches = collect(eachmatch(pattern, line))
    
    if length(matches) != 3
        error("Expected 3 vectors, found $(length(matches))")
    end
    
    parse_vec(m) = [parse(Float64, m[1]), parse(Float64, m[2]), parse(Float64, m[3])]
    
    g = parse_vec(matches[1])
    a = parse_vec(matches[2])
    m = parse_vec(matches[3])
    
    return g, a, m
end

function read_imu(ser)
    line = readline(ser)
    g, a, m = parse_imu_line(line)

    y = [a/norm(a); m/norm(m)]
    
    return g, y
end

In [ ]:
ser = open(serial_port, baud_rate)

In [ ]:
#test that we can read stuff...
line = readline(ser)

In [ ]:
#Take current attitude as identity and initialize bias with current measurement
g0, y0 = read_imu(ser)
r_N = [y0[1:3] y0[4:6]]

In [ ]:
close(ser)

In [ ]:
# Filter Initialization
x = zeros(7)
x[1:4,1] = [1.0; 0; 0; 0];
x[5:7,1] = g0;

P = zeros(6,6)
P[:,:,1] .= 0.5*I(6);

V = Diagonal([3.8e-4*ones(3); 1.8e-6*ones(3)])
W = 1e-3*I(6)

In [ ]:
ser = open(serial_port, baud_rate)
line = readline(ser)

for k = 1:600

    #Get latest measurement
    gyro, y = read_imu(ser)
    
    #Prediction
    xpred = state_prediction(x,gyro,h)
    A = state_prediction_deriv(x,gyro,h)
    Ppred = A*P*A' + V

    #Innovation
    z = y - measurement_prediction(xpred,r_N)
    C = measurement_prediction_deriv(xpred,r_N)
    S = C*Ppred*C' + W

    #Kalman Gain
    K = Ppred*C'/S

    #Update
    Δx = K*z
    ϕ = Δx[1:3]
    Δq = [sqrt(1-ϕ'*ϕ); ϕ]
    x = [L(xpred[1:4])*Δq; xpred[5:7]+Δx[4:6]]
    
    P = (I(6) - K*C)*Ppred*(I(6) - K*C)' + K*W*K'

    #Update frame
    settransform!(vis[:box1], LinearMap(QuatRotation(x[1], x[2], x[3], x[4])));
    sleep(h)
    
end

close(ser)